# Day 1 — Multi-Step Retrieval & Planner/Executor Flows

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | From one retrieval to many | Agentic RAG adds a planner that can loop — that loop is the whole new surface area |
| 2 | Real incident: Klarna's AI rollback | Simple queries matched humans; complex multi-step disputes didn't |
| 3 | New failure modes | Infinite loops, premature stops, query drift — none of these exist in a single-pass system |
| 4 | Equivalence partitioning by hop count | Same Module 4 Day 4 technique, new dimension |
| 5 | Building and tracing a 2-hop loop | See the planner's decision at every hop, not just the final answer |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Every cell runs offline — `@traceable` works without a LangSmith API key.

---

> **Where we are in the course**
> Module 4 Day 4 gave you equivalence partitioning, boundary value analysis, the coverage matrix, and hard negatives.
> Module 5 gave you the retriever/generator split, RAGAS's 4 core metrics, and LangSmith tracing for a single-pass pipeline.
> Today the pipeline gets a planner that can decide to retrieve more than once before answering — same testing toolkit, one new component to point it at.

---
## From one retrieval to many

Module 5's flow was a straight line:

```
User question -> RETRIEVER -> GENERATOR -> answer
```

Agentic RAG adds a decision point that can loop:

```
User question
    │
    ▼
PLANNER  →  do I have enough information to answer? if not, what should I search for next?
    │
    ├─ NOT ENOUGH  →  reformulate query  →  RETRIEVER  →  back to PLANNER
    │
    └─ ENOUGH      →  GENERATOR  →  final answer
```

> **Plain English:** Module 5's system was a librarian who fetches one book and writes a summary. An agentic RAG system is a research assistant who reads the first book, realizes it raises a follow-up question, goes back for a second book, and only writes the summary once they actually have what they need. The second assistant is more capable — and has far more ways to go wrong before they ever start writing.

---
## Real incident: Klarna's AI customer service rollback (May 2025)

Klarna replaced roughly 700 customer-service roles with an OpenAI-powered AI assistant, reporting in early 2024 that it handled two-thirds of chats with under-2-minute resolution times. By May 2025, CEO Sebastian Siemiatkowski publicly walked the rollout back and resumed hiring human agents. The detail that matters here: **AI matched human performance on simple queries** (order status, payment schedules) **but quality dropped noticeably on complex cases** — disputes, fraud claims, hardship cases — exactly the questions that require chaining several pieces of information together rather than answering from one lookup.

> **Why this matters for today:** "simple query" is a Module 5 problem — one retrieval, one answer. "Complex dispute" is a Module 6 problem — it needs the planner to recognize that one retrieved fact isn't enough, go fetch more, and reason across all of it. Klarna's gap between the two is the exact gap this module tests for *before* a rollout, not after.

---
## New failure modes that only exist in multi-step retrieval

| Failure mode | What it looks like |
|---|---|
| **Infinite retrieval loop** | The planner never decides it has enough information; it keeps reformulating and re-querying |
| **Premature stop** | The planner answers after 1 hop when the question genuinely needed 2-3 |
| **Query drift** | Each reformulated query moves further from the user's actual intent |
| **Reasoning chain break** | Every needed fact was retrieved correctly across hops, but combined incorrectly at the end (Day 2 builds this one) |

None of these can happen in Module 5's single-pass flow — they only exist because the system can now make a *decision* about whether to continue.

---
## Equivalence partitioning: hops required

Same technique from Module 4 Day 4, new dimension.

In [ ]:
hop_partitions = {
    "hops_required": {
        "single_hop":      "the answer is fully contained in one retrievable chunk",
        "two_hop":         "the answer requires combining facts from two separate retrievals",
        "three_plus_hop":  "the answer requires chaining three or more retrievals",
    },
}

for name, description in hop_partitions["hops_required"].items():
    print(f"[{name:<16}] {description}")

print()
print("A test suite built only from 'single_hop' cases will pass beautifully and tell you")
print("nothing about whether your planner can handle a Klarna-style dispute — the same")
print("'happy path only' trap from Module 4 Day 4, one layer up.")

---
## Building and tracing a 2-hop loop

A small knowledge base where the answer genuinely requires two hops: *"What is the cancellation fee for the product that replaced WidgetPro 2000?"* You first need to find out WHAT replaced it, then look up THAT product's fee.

In [ ]:
from langsmith import traceable

KNOWLEDGE_BASE = {
    "discontinuation": "WidgetPro 2000 was discontinued in 2023 and replaced by WidgetPro 3000.",
    "fee_3000":        "WidgetPro 3000's cancellation fee is $0 — it can be canceled anytime at no charge.",
    "fee_2000":        "WidgetPro 2000's cancellation fee was $50 before it was discontinued.",
}

@traceable(run_type="retriever")
def retrieve(query: str) -> list[str]:
    # A deliberately simple, deterministic stand-in retriever — real systems use
    # embedding similarity (Module 5 Day 2). Keeping it rule-based here makes the
    # PLANNER's looping behavior the thing under test, not the retriever's ranking.
    q = query.lower()
    if "3000" in q:
        return [KNOWLEDGE_BASE["fee_3000"]]
    if "replaced" in q or "discontinued" in q:
        return [KNOWLEDGE_BASE["discontinuation"]]
    return [KNOWLEDGE_BASE["fee_2000"]]

@traceable(run_type="chain", name="planner")
def has_enough_info(question: str, facts_so_far: list[str]) -> bool:
    # Rule-based stand-in for an LLM planner: "enough" once we have a fee figure.
    return any("fee is" in f or "fee was" in f for f in facts_so_far)

@traceable(run_type="chain", name="agentic_rag_loop")
def agentic_rag(question: str, max_hops: int = 3) -> tuple[str, list[str]]:
    facts: list[str] = []
    query = question
    for hop in range(1, max_hops + 1):
        new_facts = retrieve(query)
        facts.extend(new_facts)
        if has_enough_info(question, facts):
            break
        # Reformulate: once we know the replacement product, search for ITS fee.
        query = "WidgetPro 3000 cancellation fee" if hop == 1 else question
    return f"[{hop} hop(s)] facts used: {facts}", facts

answer, facts = agentic_rag("What is the cancellation fee for the product that replaced WidgetPro 2000?")
print(answer)

With `LANGSMITH_TRACING` enabled (set it in `.env` — see `.env.example`), this shows up at [smith.langchain.com](https://smith.langchain.com) as a single `agentic_rag_loop` trace with nested `retriever` and `planner` spans per hop — exactly the trace tree you'd inspect to tell whether a real failure was premature stopping (a `planner` span returning `True` too early) or query drift (the reformulated query in the second `retriever` span has wandered off-topic).

---
## Try It Yourself

1. Force a **premature stop** bug: change `has_enough_info()` so it returns `True` as soon as *any* fact is found, regardless of content. Run `agentic_rag()` again — what wrong answer does the 1-hop result give, and why does it look superficially plausible?
2. Force an **infinite loop** bug: change the reformulation logic so `query` never actually changes between hops. What happens when `max_hops` is reached? What should a production system do differently from this notebook's silent `for` loop exit?
3. Add a third hop to the knowledge base and a 3-hop question of your own. Update `retrieve()` and `has_enough_info()` so the loop correctly takes exactly 3 hops.

Exercise file: [`exercises/01_multistep_retrieval_exercise.md`](../exercises/01_multistep_retrieval_exercise.md)

---
## Summary

### What we built today
- The planner/executor flow that distinguishes agentic RAG from Module 5's single-pass pipeline
- A real incident (Klarna) mapped directly onto the single-hop/multi-hop quality gap
- 4 new failure modes that only exist once retrieval can loop
- An equivalence partition by `hops_required`, reusing Module 4 Day 4's technique
- A traced, working 2-hop loop you can deliberately break in 3 different ways

### Carried forward unchanged
Equivalence partitioning still works the same way it did in Module 4 Day 4 and Module 5 Day 2 — point it at a new dimension (hop count) and it surfaces the same kind of accidental happy-path-only coverage gap.

**Next:** Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing, where we build hard negatives for the failure modes that hide *inside* a clean-looking multi-hop trace.

---